# ECMWF Tropical Cyclone Processing Pipeline

This notebook demonstrates how to use the ECMWF tropical cyclone processing pipeline modules.

## Pipeline Overview

The pipeline consists of 6 main steps:
1. **Download TC Data** - Download tropical cyclone track BUFR files
2. **Extract BUFR Data** - Extract structured data from BUFR files  
3. **Transform Data** - Convert to standardized format with wind radii
4. **Download Wind Data** - Download ensemble wind forecasts
5. **Process Wind Combination** - Create wind threshold envelopes

## Output Files

1. **TC Track Data** (`*_transformed.csv`) - Individual forecast points with wind radii
2. **Individual Wind Envelopes** (`*_envelopes_individual.csv`) - Wind threshold polygons per forecast step
3. **Combined Wind Envelopes** (`*_envelopes_combined.csv`) - Combined wind threshold polygons


## Setup


In [ ]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

from numpy.f2py.crackfortran import verbose

# Import pipeline modules
from ecmwf_tc_data_downloader import download_tc_data, get_available_dates
from ecmwf_tc_data_extractor import extract_tc_data_from_file
from ecmwf_tc_data_transformer import transform_tc_data_from_file
from ecmwf_wind_data_downloader import download_ensemble_wind
from ecmwf_tc_wind_combination import process_wind_combination

In [ ]:
# Get available dates and show options
available_dates = get_available_dates()

print("Available dates:")
for i, date in enumerate(available_dates[:4]):  # First 4 dates (most recent)
    # Parse the date: YYYYMMDDHHMMSS -> YYYY-MM-DD HHZ
    date_str = date[:8]  # YYYYMMDD
    run_time = date[8:10]  # HH
    human_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"

    if i == 0:  # Most recent date
        print(f"  Today: {human_date} {run_time}Z (May not be fully published)")
    else:
        print(f"  {human_date} {run_time}Z")

print("\nAvailable run times: 00, 06, 12, 18 (UTC)")

# User configuration - modify these values
SELECTED_DATE = available_dates[1][:8]  # Second most recent date (YYYYMMDD format)
SELECTED_RUNTIME = available_dates[1][8:10]  # Run time from that date

print(f"\nSelected: {SELECTED_DATE} {SELECTED_RUNTIME}Z")


In [ ]:
SELECTED_DATE = '20251029'
SELECTED_RUNTIME = '00'  # Run time from that date

In [ ]:
# Create output directories
directories = {
    'tc_data': 'tc_data',
    'tc_extracted': 'tc_extracted', 
    'tc_transformed': 'tc_transformed',
    'wind_data': 'wind_data',
    'wind_extracted': 'wind_extracted'
}

for dir_path in directories.values():
    Path(dir_path).mkdir(exist_ok=True)

## Step 1: Download TC Data


In [ ]:
# Download TC data for selected date
tc_files = download_tc_data(
    date=SELECTED_DATE,
    run_time=SELECTED_RUNTIME,
    output_dir=directories['tc_data'],
    named_storms_only=True
)


## Step 2: Extract BUFR Data


In [ ]:
# Extract data from BUFR files
bufr_files = list(Path(directories['tc_data']).glob("*.bin"))
extraction_results = []

for bufr_file in bufr_files:
    result = extract_tc_data_from_file(
        filename=str(bufr_file),
        output_dir=directories['tc_extracted'],
        verbose=True
    )
    if result['success']:
        extraction_results.append(result)


## Step 3: Transform Data


In [ ]:
# Transform extracted data
transformation_results = []

for result in extraction_results:
    transformed_csv = transform_tc_data_from_file(
        filename=result['csv_file'],
        output_dir=directories['tc_transformed'],
        verbose=True
    )
    if transformed_csv:
        transformation_results.append({'transformed_csv': transformed_csv})


## Step 4: Download Wind Data


In [ ]:
# Download wind forecast data
forecast_hours = list(range(0, 145, 6))  # only downloading for 0h and 6h, max 0-144h every 6h

wind_files = download_ensemble_wind(
    date=f"{SELECTED_DATE[:4]}-{SELECTED_DATE[4:6]}-{SELECTED_DATE[6:]}",
    run_time=int(SELECTED_RUNTIME),
    forecast_hours=forecast_hours,
    output_dir=directories['wind_data']
)


## Step 5: Process Wind Combination


In [ ]:
# Create wind threshold envelopes
combination_result = process_wind_combination(
    tc_data_dir=Path(directories['tc_transformed']),
    wind_data_dir=Path(directories['wind_data']),
    output_dir=Path(directories['wind_extracted'])
)


## Results


In [ ]:
from visualization import show_tracks, show_tracks_with_polygons, show_individual_envelopes, show_combined_envelopes

# Check generated files
transformed_files = list(Path(directories['tc_transformed']).glob("*.csv"))
envelope_files = list(Path(directories['wind_extracted']).glob("*.csv"))

print(f"TC Track Data: {len(transformed_files)} files")
print(f"Wind Envelopes: {len(envelope_files)} files")

# Show file names
for file in transformed_files:
    print(f"  - {file.name}")
for file in envelope_files:
    print(f"  - {file.name}")


In [ ]:
# Creating TC track visualization
show_tracks(str(transformed_files[0]))

In [ ]:
# Creating TC tracks with wind polygons visualization
show_tracks_with_polygons(str(transformed_files[0]))

In [ ]:
# Creating individual wind envelopes visualizatio
individual_files = [f for f in envelope_files if 'individual' in f.name]
show_individual_envelopes(str(individual_files[0]))

In [ ]:
# Creating combined wind envelopes visualization
combined_files = [f for f in envelope_files if 'combined' in f.name]
show_combined_envelopes(str(combined_files[0]))